# 张量操作

> 张量是数据与深度学习之间的共同语言。每张图像、每个句子、每个梯度都会流经它们。

**类型：** 构建
**语言：** Python
**先修：** Phase 1, Lessons 01（线性代数直觉）、02（向量、矩阵与运算）
**时间：** ~90 分钟

## 学习目标

- 从零实现一个张量类，支持形状、步幅、reshape、transpose 和逐元素运算
- 应用广播规则，在不复制数据的情况下对不同形状的张量执行运算
- 为点积、矩阵乘法、外积和批量运算编写 einsum 表达式
- 跟踪多头注意力每一步中的精确张量形状

## 要解决的问题

你正在构建一个 transformer。前向传播看起来很清爽。你运行它，然后看到：`RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x768 and 512x768)`。你盯着这些形状。你试着做一次转置。现在它说：`Expected 4D input (got 3D input)`。你加了一个 unsqueeze。又有别的东西坏了。

形状错误是深度学习代码中最常见的 bug。它们在概念上并不难——每个操作都有自己的形状契约——但会迅速叠加。一个 transformer 会把几十个 reshape、transpose 和 broadcast 串在一起。一个轴错了，错误就会连锁扩散。更糟的是，有些形状错误根本不会抛错。它们会在错误的维度上广播，或在错误的轴上求和，静悄悄地产生垃圾结果。

矩阵能处理两组对象之间的成对关系。真实数据并不适合塞进二维里。一批 32 张 224x224 的 RGB 图像是一个 4D 张量：`(32, 3, 224, 224)`。带 12 个头的自注意力也是 4D：`(batch, heads, seq_len, head_dim)`。你需要一种能推广到任意维度的数据结构，并且它的操作能在所有维度上干净地组合。这种结构就是张量。掌握它的操作，形状错误就会变得很容易调试。

## 核心概念

### 什么是张量

张量是一个具有统一数据类型的多维数字数组。维度数量叫作 **rank**（或 **order**）。每个维度是一条 **axis**。**shape** 是一个元组，列出每条轴上的大小。

```mermaid
graph LR
    S["Scalar<br/>rank 0<br/>shape: ()"] --> V["Vector<br/>rank 1<br/>shape: (3,)"]
    V --> M["Matrix<br/>rank 2<br/>shape: (2,3)"]
    M --> T3["3D Tensor<br/>rank 3<br/>shape: (2,2,2)"]
    T3 --> T4["4D Tensor<br/>rank 4<br/>shape: (B,C,H,W)"]
```

总元素数 = 所有大小的乘积。形状 `(2, 3, 4)` 包含 `2 * 3 * 4 = 24` 个元素。

### 深度学习中的张量形状

按照惯例，不同数据类型会映射到特定的张量形状。

```mermaid
graph TD
    subgraph Vision
        V1["(B, C, H, W)<br/>32, 3, 224, 224"]
    end
    subgraph NLP
        N1["(B, T, D)<br/>16, 128, 768"]
    end
    subgraph Attention
        A1["(B, H, T, D)<br/>16, 12, 128, 64"]
    end
    subgraph Weights
        W1["Linear: (out, in)<br/>Conv2D: (out_c, in_c, kH, kW)<br/>Embedding: (vocab, dim)"]
    end
```

PyTorch 使用 NCHW（channels-first）。TensorFlow 默认使用 NHWC（channels-last）。布局不匹配会导致悄无声息的性能下降或错误。

### 内存布局如何工作

内存中的 2D 数组其实是一维字节序列。**strides** 告诉你沿每条轴移动一步需要跳过多少个元素。

```mermaid
graph LR
    subgraph "Row-major (C order)"
        R["a b c d e f<br/>strides: (3, 1)"]
    end
    subgraph "Column-major (F order)"
        C["a d b e c f<br/>strides: (1, 2)"]
    end
```

Transpose 不会移动数据。它只是交换 strides，让张量变成 **non-contiguous**——一行里的元素在内存中不再相邻。

### 广播规则

广播让你可以在不复制数据的情况下，对不同形状的张量执行运算。从右侧对齐形状。两个维度相等，或其中一个是 1 时，它们就是兼容的。维度更少的张量会在左侧补 1。

```text
Tensor A:     (8, 1, 6, 1)
Tensor B:        (7, 1, 5)
Padded B:     (1, 7, 1, 5)
Result:       (8, 7, 6, 5)
```

### Einsum：通用张量操作

爱因斯坦求和用字母标记每条轴。出现在输入中、但没有出现在输出中的轴会被求和。输入和输出中都出现的轴会被保留。

```mermaid
graph LR
    subgraph "matmul: ik,kj -> ij"
        A["A(I,K)"] --> |"sum over k"| C["C(I,J)"]
        B["B(K,J)"] --> |"sum over k"| C
    end
```

关键模式：`i,i->`（点积）、`i,j->ij`（外积）、`ii->`（迹）、`ij->ji`（转置）、`bij,bjk->bik`（批量矩阵乘法）、`bhtd,bhsd->bhts`（注意力分数）。

## 动手实现

代码位于 `code/tensors.py`。每一步都会引用其中的实现。

### 第 1 步：张量存储与步幅

张量存储一份扁平数字列表，以及形状元数据。Strides 告诉索引逻辑如何把多维索引映射到扁平位置。


In [ ]:
class Tensor:
    def __init__(self, data, shape=None):
        if isinstance(data, (list, tuple)):
            self._data, self._shape = self._flatten_nested(data)
        elif isinstance(data, np.ndarray):
            self._data = data.flatten().tolist()
            self._shape = tuple(data.shape)
        else:
            self._data = [data]
            self._shape = ()

        if shape is not None:
            total = reduce(lambda a, b: a * b, shape, 1)
            if total != len(self._data):
                raise ValueError(
                    f"Cannot reshape {len(self._data)} elements into shape {shape}"
                )
            self._shape = tuple(shape)

        self._strides = self._compute_strides(self._shape)

    @staticmethod
    def _compute_strides(shape):
        if len(shape) == 0:
            return ()
        strides = [1] * len(shape)
        for i in range(len(shape) - 2, -1, -1):
            strides[i] = strides[i + 1] * shape[i + 1]
        return tuple(strides)


对于形状 `(3, 4)`，strides 是 `(4, 1)`——前进一行跳过 4 个元素，前进一列跳过 1 个元素。

### 第 2 步：Reshape、squeeze、unsqueeze

Reshape 会改变形状，但不改变元素顺序。元素总数必须保持不变。使用 `-1` 可以让某一个维度的大小自动推断。


In [ ]:
t = Tensor(list(range(12)), shape=(2, 6))
r = t.reshape((3, 4))
r = t.reshape((-1, 3))


Squeeze 会移除大小为 1 的轴。Unsqueeze 会插入一个轴。Unsqueeze 对广播非常关键——把偏置向量 `(D,)` 加到一个 batch `(B, T, D)` 上时，需要先 unsqueeze 成 `(1, 1, D)`。


In [ ]:
t = Tensor(list(range(6)), shape=(1, 3, 1, 2))
s = t.squeeze()
v = Tensor([1, 2, 3])
u = v.unsqueeze(0)


### 第 3 步：Transpose 与 permute

Transpose 交换两条轴。Permute 重排所有轴。这就是你在 NCHW 和 NHWC 之间转换的方式。


In [ ]:
mat = Tensor(list(range(6)), shape=(2, 3))
tr = mat.transpose(0, 1)

t4d = Tensor(list(range(24)), shape=(1, 2, 3, 4))
perm = t4d.permute((0, 2, 3, 1))


在 transpose 或 permute 之后，张量在内存中是 non-contiguous 的。在 PyTorch 中，`view` 会在 non-contiguous 张量上失败——请使用 `reshape`，或先调用 `.contiguous()`。

### 第 4 步：逐元素运算与归约

逐元素操作（加、乘、减）会独立作用在每个元素上，并保持形状不变。归约（sum、mean、max）会折叠一条或多条轴。


In [ ]:
a = Tensor([[1, 2], [3, 4]])
b = Tensor([[10, 20], [30, 40]])
c = a + b
d = a * 2
s = a.sum(axis=0)


CNN 中的全局平均池化：`(B, C, H, W).mean(axis=[2, 3])` 会产生 `(B, C)`。NLP 中的序列平均池化：`(B, T, D).mean(axis=1)` 会产生 `(B, D)`。

### 第 5 步：使用 NumPy 进行广播

`tensors.py` 中的 `demo_broadcasting_numpy()` 函数展示了核心模式。


In [ ]:
activations = np.random.randn(4, 3)
bias = np.array([0.1, 0.2, 0.3])
result = activations + bias

images = np.random.randn(2, 3, 4, 4)
scale = np.array([0.5, 1.0, 1.5]).reshape(1, 3, 1, 1)
result = images * scale

a = np.array([1, 2, 3]).reshape(-1, 1)
b = np.array([10, 20, 30, 40]).reshape(1, -1)
outer = a * b


通过广播计算成对距离：把 `(M, 2)` reshape 为 `(M, 1, 2)`，把 `(N, 2)` reshape 为 `(1, N, 2)`，相减、平方、沿最后一条轴求和，再开平方。结果是 `(M, N)`。

### 第 6 步：Einsum 运算

`demo_einsum()` 和 `demo_einsum_gallery()` 函数会走过每一种常见模式。


In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
dot = np.einsum("i,i->", a, b)

A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
B = np.array([[7, 8, 9], [10, 11, 12]], dtype=float)
matmul = np.einsum("ik,kj->ij", A, B)

batch_A = np.random.randn(4, 3, 5)
batch_B = np.random.randn(4, 5, 2)
batch_mm = np.einsum("bij,bjk->bik", batch_A, batch_B)


一次 contraction 的计算成本是所有索引大小的乘积（包括保留的索引和被求和的索引）。对于 B=32、I=128、J=64、K=128 的 `bij,bjk->bik`：`32 * 128 * 64 * 128 = 33,554,432` 次乘加。

### 第 7 步：通过 einsum 实现注意力机制

`demo_attention_einsum()` 函数从头到尾实现了多头注意力。


In [ ]:
B, H, T, D = 2, 4, 8, 16
E = H * D

X = np.random.randn(B, T, E)
W_q = np.random.randn(E, E) * 0.02

Q = np.einsum("bte,ek->btk", X, W_q)
Q = Q.reshape(B, T, H, D).transpose(0, 2, 1, 3)

scores = np.einsum("bhtd,bhsd->bhts", Q, K) / np.sqrt(D)
weights = softmax(scores, axis=-1)
attn_output = np.einsum("bhts,bhsd->bhtd", weights, V)

concat = attn_output.transpose(0, 2, 1, 3).reshape(B, T, E)
output = np.einsum("bte,ek->btk", concat, W_o)


每一步都是张量操作：投影（通过 einsum 做 matmul）、拆分头（reshape + transpose）、attention scores（通过 einsum 做批量矩阵乘法）、加权和（通过 einsum 做批量矩阵乘法）、合并头（transpose + reshape）、输出投影（通过 einsum 做 matmul）。

## 实际使用

### 从零实现 vs NumPy

| 操作 | 从零实现（Tensor 类） | NumPy |
|---|---|---|
| 创建 | `Tensor([[1,2],[3,4]])` | `np.array([[1,2],[3,4]])` |
| Reshape | `t.reshape((3,4))` | `a.reshape(3,4)` |
| Transpose | `t.transpose(0,1)` | `a.T` 或 `a.transpose(0,1)` |
| Squeeze | `t.squeeze(0)` | `np.squeeze(a, 0)` |
| Sum | `t.sum(axis=0)` | `a.sum(axis=0)` |
| Einsum | 不适用 | `np.einsum("ij,jk->ik", a, b)` |

### 从零实现 vs PyTorch


In [ ]:
import torch

t = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
t.shape
t.stride()
t.is_contiguous()

t.reshape(3, 2)
t.unsqueeze(0)
t.transpose(0, 1)
t.transpose(0, 1).contiguous()

torch.einsum("ik,kj->ij", A, B)


PyTorch 加入了 autograd、GPU 支持和优化过的 BLAS kernels。形状语义是完全一样的。如果你理解从零实现版本，PyTorch 的形状错误就会变得可读。

### 每个神经网络层都是张量操作

| 操作 | 张量形式 | Einsum |
|---|---|---|
| 线性层 | `Y = X @ W.T + b` | `"bd,od->bo"` + 偏置 |
| 注意力 QKV | `Q = X @ W_q` | `"btd,dh->bth"` |
| 注意力分数 | `Q @ K.T / sqrt(d)` | `"bhtd,bhsd->bhts"` |
| 注意力输出 | `softmax(scores) @ V` | `"bhts,bhsd->bhtd"` |
| 批归一化 | `(X - mu) / sigma * gamma` | 逐元素 + 广播 |
| Softmax | `exp(x) / sum(exp(x))` | 逐元素 + 归约 |

## 交付成果

本课会产出两个可复用提示词：

1. **`outputs/prompt-tensor-shapes.md`**——一个用于系统化调试张量形状不匹配的提示词。包含每种常见操作（matmul、broadcast、cat、Linear、Conv2d、BatchNorm、softmax）的决策表，以及修复查找表。

2. **`outputs/prompt-tensor-debugger.md`**——一个逐步调试提示词。当形状错误卡住你时，可以把它粘贴进任何 AI assistant。输入错误消息和你的张量形状，就能得到精确修复。

## 练习

1. **简单——Reshape 往返。** 取一个形状为 `(2, 3, 4)` 的张量。把它 reshape 为 `(6, 4)`，再 reshape 为 `(24,)`，然后回到 `(2, 3, 4)`。通过打印扁平数据，验证每一步都保留了元素顺序。

2. **中等——实现广播。** 给 `Tensor` 类扩展一个 `broadcast_to(shape)` 方法，把大小为 1 的维度扩展到目标形状。然后修改 `_elementwise_op`，让它在执行运算前自动广播。用形状 `(3, 1)` 和 `(1, 4)` 测试，结果应为 `(3, 4)`。

3. **困难——从零构建 einsum。** 实现一个基础的 `einsum(subscripts, *tensors)` 函数，至少处理：点积（`i,i->`）、矩阵乘法（`ij,jk->ik`）、外积（`i,j->ij`）和转置（`ij->ji`）。解析 subscript 字符串，识别被 contraction 的索引，并遍历所有索引组合。把你的结果与 `np.einsum` 对比。

4. **困难——注意力形状追踪器。** 写一个函数，接收 `batch_size`、`seq_len`、`embed_dim` 和 `num_heads` 作为输入，并打印多头注意力每一步的精确形状：输入、Q/K/V 投影、拆分头、注意力分数、softmax 权重、加权和、合并头、输出投影。对照 `demo_attention_einsum()` 的输出进行验证。

## 关键术语

| 术语 | 人们常说 | 实际含义 |
|---|---|---|
| Tensor | “更高维的 matrix” | 具有统一类型以及明确定义的 shape、strides 和 operations 的多维数组 |
| Rank | “维度数量” | 轴的数量。矩阵的 rank 是 2，不等同于线性代数里的矩阵秩 |
| Shape | “张量的大小” | 一个元组，列出每条轴上的大小。`(2, 3)` 表示 2 行、3 列 |
| Stride | “内存是怎么排布的” | 沿每条轴前进一个位置时需要跳过的元素数量 |
| Broadcasting | “形状不同时它也能工作” | 一组严格规则：从右侧对齐，维度必须相等，或其中一个必须是 1 |
| Contiguous | “张量是正常的” | 元素按逻辑布局连续存储在内存中，没有间隙或重排 |
| Einsum | “一种花哨的 matmul 写法” | 一种通用记法，可以用一行表达任何 tensor contraction、outer product、trace 或 transpose |
| View | “和 reshape 一样” | 与原张量共享同一块内存缓冲区，但使用不同 shape/stride 元数据的张量。会在 non-contiguous 数据上失败 |
| Contraction | “对某个索引求和” | 张量之间共享索引先相乘再求和、产生更低 rank 结果的一般操作 |
| NCHW / NHWC | “PyTorch vs TensorFlow 格式” | 图像张量的内存布局约定。NCHW 把 channels 放在空间维度之前，NHWC 把它们放在之后 |

## 延伸阅读

- [NumPy Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)——带可视化示例的规范规则
- [PyTorch Tensor Views](https://pytorch.org/docs/stable/tensor_view.html)——什么时候 view 可用，什么时候会复制
- [einops](https://github.com/arogozhnikov/einops)——一个让张量 reshape 更可读、更安全的库
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)——可视化 attention 中流动的张量形状
- [Einstein Summation in NumPy](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)——带示例的完整 einsum 文档
